# 第 7 章：指令微调（Instruction Fine-tuning）

前几章的 GPT 只会「续写」。本章让它学会**遵循指令**——给定「翻译这个句子」，它能输出翻译结果，而非继续编故事。

这是把通用语言模型变成**实用助手**的关键一步，也是 ChatGPT 等模型的核心训练阶段之一。

## 核心思路

1. **指令数据格式**：用 Alpaca 模板把「指令 + 输入」组织成 prompt，模型学习生成「响应」
2. **loss masking（本章灵魂）**：只对「响应」部分算 loss，指令部分被 mask 掉（否则模型会学着复述指令）
3. **微调训练**：复用 GPT 预训练架构，在指令数据上继续训练

> 主线用自造中文指令 demo 数据（情感识别/翻译/问答）验证流程。真实场景用 Alpaca/OASST 等数万条指令数据集。

## 1. Alpaca 指令格式

Alpaca（斯坦福）提出了一套标准化的指令模板，用分隔符区分「指令/输入/响应」三段：

```
### Instruction:
翻译成英文
### Input:
你好
### Response:
 Hello
```

> 分隔符让模型学会在 `### Response:` 之后开始生成回答。

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import tiktoken
from src.gpt import GPTModel, GPT_CONFIG_124M

# 自造中文指令数据（instruction + input + output 三元组）
INSTRUCTION_DATA = [
    {"instruction": "识别情感", "input": "今天天气真好", "output": " 正面"},
    {"instruction": "识别情感", "input": "太让人失望了", "output": " 负面"},
    {"instruction": "识别情感", "input": "电影非常精彩", "output": " 正面"},
    {"instruction": "识别情感", "input": "服务态度很差", "output": " 负面"},
    {"instruction": "翻译成英文", "input": "你好", "output": " Hello"},
    {"instruction": "翻译成英文", "input": "谢谢", "output": " Thank you"},
    {"instruction": "翻译成英文", "input": "再见", "output": " Goodbye"},
    {"instruction": "回答问题", "input": "法国首都是哪", "output": " 巴黎"},
    {"instruction": "回答问题", "input": "地球绕着什么转", "output": " 太阳"},
    {"instruction": "回答问题", "input": "一年有几个月", "output": " 十二"},
] * 3  # 复制增加样本量


def format_prompt(entry):
    """Alpaca 模板：把指令和输入拼成 prompt。"""
    return (
        f"### Instruction:\n{entry['instruction']}\n"
        f"### Input:\n{entry['input']}\n"
        f"### Response:\n"
    )


# 看一条样例
sample = INSTRUCTION_DATA[0]
print("prompt:")
print(format_prompt(sample) + sample['output'])

## 2. loss masking（本章核心）

指令微调的关键：**只对 Response 部分算 loss**，Instruction 部分被 mask 成 `-100`。

- 如果对整个序列算 loss，模型会花力气学「复述指令」，浪费容量
- 只算 Response 的 loss，模型聚焦于「给定指令，生成正确回答」这个目标

> PyTorch 的 `cross_entropy(ignore_index=-100)` 会自动跳过 `-100` 的位置。

In [ ]:
def custom_collate(batch_entries, tokenizer, max_len=64, pad_id=50256, ignore_index=-100):
    """整理批次：拼 prompt+response，构造 loss mask。

    关键：targets 中 prompt 对应位置置 -100，只对 response 算 loss。
    """
    batch_inputs, batch_targets = [], []
    for entry in batch_entries:
        prompt_ids = tokenizer.encode(format_prompt(entry))
        resp_ids = tokenizer.encode(entry["output"])
        input_ids = prompt_ids + resp_ids
        # targets = input_ids 右移一位（预测下一个 token）
        targets = input_ids[1:] + [tokenizer.eot_token]
        # 截断到 max_len
        input_ids = input_ids[:max_len]
        targets = targets[:max_len]
        # ★ mask 掉 prompt 部分（targets 中对应 prompt 的位置置 -100）
        n_prompt = len(prompt_ids)
        for i in range(min(n_prompt, len(targets))):
            targets[i] = ignore_index
        # pad 到 max_len
        input_ids = input_ids + [pad_id] * (max_len - len(input_ids))
        targets = targets + [ignore_index] * (max_len - len(targets))
        batch_inputs.append(input_ids)
        batch_targets.append(targets)
    return torch.tensor(batch_inputs), torch.tensor(batch_targets)


tok = tiktoken.get_encoding("gpt2")
x, y = custom_collate(INSTRUCTION_DATA[:2], tok, max_len=48)
print(f"batch: x {tuple(x.shape)}, y {tuple(y.shape)}")
n_masked = (y[0] == -100).sum().item()
n_calc = (y[0] != -100).sum().item()
print(f"样本0: mask {n_masked} 位置(prompt+pad), 计算 loss {n_calc} 位置(response)")

## 3. 指令微调训练

和 ch06 一样，冻结 backbone，只训练最后块 + final_norm + 输出头。区别是 loss：这里对**序列做下一步预测**（Response 部分），而非分类。

> 注意：微调时 out_head 保持原样（vocab_size），因为要生成文本 token，不是分类。

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = dict(GPT_CONFIG_124M)
cfg.update({"emb_dim": 128, "n_layers": 2, "n_heads": 4, "context_length": 64})

torch.manual_seed(123)
model = GPTModel(cfg)
# 冻结 backbone，只训最后块 + norm + 输出头（out_head 保持 vocab_size 用于生成）
for p in model.parameters():
    p.requires_grad = False
for p in model.trf_blocks[-1].parameters():
    p.requires_grad = True
for p in model.final_norm.parameters():
    p.requires_grad = True
for p in model.out_head.parameters():
    p.requires_grad = True

model.to(device)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=5e-4, weight_decay=0.1
)

dataloader = DataLoader(
    INSTRUCTION_DATA, batch_size=4, shuffle=True,
    collate_fn=lambda b: custom_collate(b, tok, max_len=cfg["context_length"]),
)

model.train()
for epoch in range(10):
    total = 0; n = 0
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        # flatten 后 cross_entropy 自动忽略 -100（prompt 部分）
        loss = F.cross_entropy(logits.flatten(0, 1), y.flatten(), ignore_index=-100)
        loss.backward()
        optimizer.step()
        total += loss.item(); n += 1
    if epoch % 2 == 0 or epoch == 9:
        print(f"epoch {epoch:2d}: loss {total/n:.4f}")

## 4. 生成测试：给定指令，模型续写回答

In [ ]:
def generate_response(model, entry, tokenizer, max_new_tokens=10, device="cpu"):
    """给定指令，让模型生成 Response。"""
    model.eval()
    prompt = format_prompt(entry)
    idx = torch.tensor([tokenizer.encode(prompt)]).to(device)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -cfg["context_length"]:]
            logits = model(idx_cond)
            next_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            idx = torch.cat([idx, next_id], dim=1)
            # 遇到结束符停止
            if next_id.item() == tokenizer.eot_token:
                break
    generated = tokenizer.decode(idx[0].tolist())
    # 提取 Response 部分
    response = generated.split("### Response:\n")[-1]
    return response.strip()


# 测试几条指令
test_cases = [
    {"instruction": "识别情感", "input": "今天天气真好"},
    {"instruction": "翻译成英文", "input": "你好"},
    {"instruction": "回答问题", "input": "法国首都是哪"},
]
print("生成测试（demo 数据，未加载预训练权重）：")
for entry in test_cases:
    resp = generate_response(model, entry, tok, device=device)
    print(f"  [{entry['instruction']}] {entry['input']!r} → {resp!r}")
print("\n💡 demo 数据量小、模型未预训练，输出可能不完美。")
print("   真实场景加载 OpenAI 权重 + 数万条指令数据后，能生成高质量回答。")